# Week 4: American Put Baseline Pricer

CRR binomial tree implementation for European and American puts, with convergence analysis, price surface, and early-exercise boundary visualization.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from american_put import crr_put_price, price_grid, crr_put_with_boundary, convergence_table

# Create figures directory if needed
os.makedirs('figures', exist_ok=True)

## 1. Inputs

In [ ]:
# Define base parameters
S0 = 100          # Spot price ($)
K = 100           # Strike price ($)
T = 1.0           # Time to maturity (years)
r = 0.05          # Risk-free rate (5% annual)
sigma = 0.25      # Volatility (25% annual)
steps = 500       # Default tree depth

print("Parameters:")
print(f"  Spot price (S0)        : ${S0}")
print(f"  Strike price (K)       : ${K}")
print(f"  Time to maturity (T)   : {T} years")
print(f"  Risk-free rate (r)     : {r*100:.1f}%")
print(f"  Volatility (sigma)     : {sigma*100:.1f}%")
print(f"  Tree steps (default)   : {steps}")

## 2. Implementation Summary

The Cox-Ross-Rubinstein (CRR) binomial tree models the asset price path with discrete up and down moves:

**CRR Parameters:**
- $\Delta t = T / N$  (step length)
- $u = e^{\sigma\sqrt{\Delta t}}$  (up factor)
- $d = 1/u$  (down factor)
- $p = \frac{e^{r\Delta t} - d}{u - d}$  (risk-neutral probability)

**Backward Induction:**
1. Terminal payoff: $\max(K - S, 0)$ at all final nodes
2. For each earlier node: $V = e^{-r\Delta t}[p \cdot V_{up} + (1-p) \cdot V_{down}]$
3. American feature: $V = \max(V_{continuation}, K - S)$ at each node

The American option's early-exercise feature adds value because the holder can exercise whenever beneficial.

## 3. Core Prices: European vs American

In [ ]:
# Price both European and American puts
euro_price = crr_put_price(S0, K, T, r, sigma, steps, american=False)
amer_price = crr_put_price(S0, K, T, r, sigma, steps, american=True)
premium = amer_price - euro_price

print(f"European put price      : ${euro_price:.6f}")
print(f"American put price      : ${amer_price:.6f}")
print(f"Early-exercise premium  : ${premium:.6f} ({premium/euro_price*100:.2f}% of European)")

## 4. Sanity Tests

Finance-based checks that catch common implementation bugs:

In [ ]:
# Test 1: American >= European
test1_euro = crr_put_price(100, 105, 1.0, 0.05, 0.25, 500, american=False)
test1_amer = crr_put_price(100, 105, 1.0, 0.05, 0.25, 500, american=True)
test1_pass = test1_amer >= test1_euro
print(f"✓ Test 1: American >= European")
print(f"  {test1_amer:.6f} >= {test1_euro:.6f} : {test1_pass}")
print()

# Test 2: Put value falls as spot rises
test2_low = crr_put_price(80, 100, 1.0, 0.05, 0.25, 500, american=True)
test2_high = crr_put_price(120, 100, 1.0, 0.05, 0.25, 500, american=True)
test2_pass = test2_low > test2_high
print(f"✓ Test 2: Value falls as spot rises")
print(f"  ${test2_low:.6f} > ${test2_high:.6f} : {test2_pass}")
print()

# Test 3: More volatility is not cheaper
test3_low_vol = crr_put_price(100, 100, 1.0, 0.05, 0.15, 500, american=True)
test3_high_vol = crr_put_price(100, 100, 1.0, 0.05, 0.35, 500, american=True)
test3_pass = test3_high_vol >= test3_low_vol
print(f"✓ Test 3: Value increases with volatility")
print(f"  ${test3_high_vol:.6f} >= ${test3_low_vol:.6f} : {test3_pass}")
print()

all_pass = test1_pass and test2_pass and test3_pass
print(f"All tests passed: {all_pass}")

## 5. Convergence Table

In [ ]:
# Convergence analysis: how price changes with tree depth
table = convergence_table(S0, K, T, r, sigma)

print("American put price convergence:")
print("Steps     Price       Change from previous")
print("-" * 45)

prev_price = None
for steps_i, price in table:
    if prev_price is None:
        change = "-"
    else:
        change = f"{abs(price - prev_price):.8f}"
    print(f"{steps_i:4d}      {price:.8f}    {change}")
    prev_price = price

print()
prices = [p for _, p in table]
print(f"Final price (1000 steps)   : ${prices[-1]:.8f}")
print(f"Change 500→1000 steps      : ${abs(prices[-1] - prices[-2]):.8f}")
print()
print("Interpretation: Prices stabilize around 500 steps. Using 500 steps")
print("for baseline pricing provides good balance between accuracy and speed.")

## 6. Price Surface

In [ ]:
# Generate 2D price grid over spot and maturity
print("Generating price grid... (this takes ~30 seconds)")
spots, maturities, prices = price_grid(K=K, r=r, sigma=sigma, steps=300)
print("Grid complete.")

# Create 3D surface plot
S_mesh, T_mesh = np.meshgrid(spots, maturities)

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_mesh, T_mesh, prices, cmap="viridis", linewidth=0, alpha=0.9)

ax.set_xlabel("Spot S ($)", fontsize=11, labelpad=10)
ax.set_ylabel("Time to maturity T (years)", fontsize=11, labelpad=10)
ax.set_zlabel("American put price ($)", fontsize=11, labelpad=10)
ax.set_title("CRR American Put Price Surface\n(K=100, r=5%, σ=25%)", fontsize=13, pad=20)

fig.colorbar(surf, ax=ax, pad=0.1, label="Price ($)")
plt.tight_layout()
plt.savefig("figures/price_surface.png", dpi=160, bbox_inches="tight")
plt.show()

print("✓ Saved figures/price_surface.png")

**Surface interpretation:**

- **Far OTM (S >> K):** Price → 0. The put is nearly worthless when spot is far above strike.
- **Deep ITM (S << K):** Price → K − S. The put behaves like a forward contract to sell at K.
- **Near K:** Strong curvature. This is where uncertainty (volatility) has the most impact.
- **Maturity effect:** Longer maturity adds optionality, but early exercise creates interesting nonlinearity.

## 7. Early-Exercise Boundary

In [ ]:
# Find and plot the exercise boundary
price_boundary, boundary = crr_put_with_boundary(S0, K, T, r, sigma, 500)

times = [t for t, _ in boundary]
spots_boundary = [s for _, s in boundary]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(times, spots_boundary, marker="o", linewidth=2.5, markersize=5, 
        label="Exercise boundary", color="#1f77b4")
ax.axhline(K, color="gray", linestyle="--", linewidth=1.5, alpha=0.7, label=f"Strike K = ${K}")

ax.set_xlabel("Time to maturity (years)", fontsize=12)
ax.set_ylabel("Spot price ($)", fontsize=12)
ax.set_title("American Put Early-Exercise Boundary\n(S₀=100, K=100, T=1, r=5%, σ=25%)", fontsize=13)
ax.legend(fontsize=11, loc="upper right")
ax.grid(True, alpha=0.3)
ax.set_xlim(0, T)

plt.tight_layout()
plt.savefig("figures/exercise_boundary.png", dpi=160, bbox_inches="tight")
plt.show()

print("✓ Saved figures/exercise_boundary.png")

## 8. Reflection: Where Does Early Exercise Appear?

### Boundary Definition
The exercise boundary marks the highest spot price $S^*_t$ at each time $t$ where immediate exercise is optimal:
- **Below** the boundary ($S < S^*_t$): Immediate exercise is optimal.
- **Above** the boundary ($S > S^*_t$): Holding the option is optimal.

### Why Is This Financially Sensible?

#### 1. **Moneyness**
Early exercise is most attractive when the option is deep in-the-money (S far below K). When ITM, the put has large intrinsic value (K - S). The holder can lock this in now instead of waiting for maturity.

#### 2. **Interest Rates & Time Value**
With positive interest rates (r > 0), the holder benefits from receiving the strike K sooner. If the option is ITM, exercising now captures K immediately rather than waiting and receiving it later. The present value of K today exceeds the present value of K at maturity.

#### 3. **Boundary Shape Over Time**
- **Far from maturity:** The boundary sits well below the strike. There is time value; the holder prefers to keep optionality.
- **Near maturity (T → 0):** The boundary tightens toward the strike. Time value evaporates. For very deep ITM puts, exercise becomes irresistible.

#### 4. **European Comparison**
A European put cannot exercise early—it must hold to maturity, even if deep ITM. The American put's early-exercise flexibility is what creates the premium over the European price. This premium is largest for puts with positive interest rates and high volatility.

### Observed Pattern
The boundary shows:
- **Discrete staircase:** The tree has only certain discrete spot prices, so the boundary is step-like. This is correct and expected.
- **Monotonic:** As time passes (moving right), the boundary generally moves downward toward S = K, reflecting reduced uncertainty.
- **Never negative:** Spot prices remain positive, and the boundary stays above zero.

This exercise policy is exactly what an optimal American put holder would follow, making it the correct baseline for later ML and RL work.

## Summary

✓ **Implemented** CRR binomial tree for European and American puts  
✓ **Priced** both styles at baseline parameters (early-exercise premium quantified)  
✓ **Tested** with finance-based sanity checks (all passed)  
✓ **Analyzed** convergence (500 steps sufficient)  
✓ **Visualized** price surface and exercise boundary  
✓ **Explained** the financial intuition behind early exercise  

This baseline is now ready to serve as:
- **Target labels** for Week 6 neural network training
- **Benchmark** for Week 7–8 reinforcement learning
- **Reference prices** for the final report